In [1]:
%cd /kaggle/working
!git clone https://github.com/shujaabbasn/ATML.git
%cd /kaggle/working/ATML/pa4
!python -m pip install -q -r requirements.txt
!nvidia-smi

/kaggle/working
Cloning into 'ATML'...
remote: Enumerating objects: 1163, done.
remote: Counting objects: 100% (611/611), done.
remote: Compressing objects: 100% (392/392), done.
remote: Total 1163 (delta 244), reused 578 (delta 215), pack-reused 552 (from 1)
Receiving objects: 100% (1163/1163), 35.73 MiB | 31.21 MiB/s, done.
Resolving deltas: 100% (474/474), done.
/kaggle/working/ATML/pa4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompati

In [2]:
# staff data repo (scripts + bundled files), then merge anything attached from earlier sessions
!git clone -q https://github.com/AbDu11aHHH/atml-pa4-datasets.git data_repo
!python kaggle_runs/attach_inputs.py
# skips files that already exist, so with data_prepare attached this is only a check (C4 streaming is slow otherwise)
!cd data_repo && python scripts/setup_data.py --public && python scripts/check_setup.py

unzip /kaggle/input/notebooks/shujabbasn/data-prepare-run/data_repo.zip
unzip /kaggle/input/notebooks/shujabbasn/task2-attention-mqar-run/task2_ab_results.zip
copy /kaggle/input/notebooks/shujabbasn/data-prepare-run/ATML/pa4/data_repo/data -> data_repo/data
copy /kaggle/input/notebooks/shujabbasn/task2-attention-mqar-run/ATML/pa4/data_repo/data -> data_repo/data
copy /kaggle/input/notebooks/shujabbasn/task2-attention-mqar-run/ATML/pa4/results -> results
merged 5 inputs
revision lock present
tokenizer_config.json: 7.23kB [00:00, 32.0MB/s]
vocab.json: 2.78MB [00:00, 49.5MB/s]
merges.txt: 1.67MB [00:00, 144MB/s]
tokenizer.json: 7.03MB [00:00, 201MB/s]
Public data available locally. Preserve data/local_source_revisions.json for reproducibility.
OK C4 calibration 128
OK C4 development 32
OK C4 validation windows 177
OK WikiText-2 test windows 437
OK HellaSwag examples 2000
OK GSM8K train 500
OK GSM8K test 60
OK prefix banks
OK source snapshot lock
PASS: public assets prepared locally. Model

In [3]:
# attach task2_attention_mqar_run as an input: its 39 finished runs come in through attach_inputs above
# the first three hybrid runs stored a running SUM in fp16, it passed 65504 and went NaN, so only they are redone
!rm -rf results/task2/part_b/hybrid_K16_mixed_seed*
!ls results/task2/part_b

config_used.json	    linear_K4_mixed_seed6305
dense_K16_mixed_seed6304    linear_K4_mixed_seed6306
dense_K16_mixed_seed6305    linear_K8_mixed_seed6304
dense_K16_mixed_seed6306    linear_K8_mixed_seed6305
dense_K16_near_seed6304     linear_K8_mixed_seed6306
dense_K16_near_seed6305     local_K16_mixed_seed6304
dense_K16_near_seed6306     local_K16_mixed_seed6305
dense_K4_mixed_seed6304     local_K16_mixed_seed6306
dense_K4_mixed_seed6305     local_K16_near_seed6304
dense_K4_mixed_seed6306     local_K16_near_seed6305
dense_K8_mixed_seed6304     local_K16_near_seed6306
dense_K8_mixed_seed6305     local_K4_near_seed6304
dense_K8_mixed_seed6306     local_K4_near_seed6305
dilated_K16_mixed_seed6304  local_K4_near_seed6306
dilated_K16_mixed_seed6305  local_K8_near_seed6304
dilated_K16_mixed_seed6306  local_K8_near_seed6305
dilated_K4_mixed_seed6304   local_K8_near_seed6306
dilated_K4_mixed_seed6305   summary.json
dilated_K4_mixed_seed6306   train_stream_K16_mixed.npz
inference_sweep.json	    

In [4]:
# retrains only the three hybrid seeds (state now stored as a running mean in fp16), the other runs are not touched
!python -m task2_attention.mqar --arch hybrid

/kaggle/working/ATML/pa4/task2_attention/mqar.py:444: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  log.append({"step":step,"loss":float(loss),"grad_norm":grad_norm,"lr":scheduler.get_last_lr()[0]})
hybrid_K16_mixed_seed6304 update 500 dev acc 0.0158
hybrid_K16_mixed_seed6304 update 1000 dev acc 0.0174
hybrid_K16_mixed_seed6304 update 1500 dev acc 0.0149
hybrid_K16_mixed_seed6304 update 2000 dev acc 0.0152
hybrid_K16_mixed_seed6304 update 3000 dev acc 0.0167
hybrid_K16_mixed_seed6304 update 4500 dev acc 0.0163
hybrid_K16_mixed_seed6305 update 500 dev acc 0.0153
hybrid_K16_mixed_seed6305 update 1000 dev acc 0.0172
hybrid_K16_mixed_seed6305 update 1500 dev acc 0.0151
hybrid_K16_mixed_seed6305 update 2000 dev acc 0.0159
hybrid_K16_mixed_seed6305 update 3000 dev acc 0.0154
hybrid_K16_mixed_seed6305 

In [5]:
# forward-only sweep again so its hybrid row uses the same code as the trained models
!python -m task2_attention.mqar --inference_sweep
!python -m task2_attention.mqar --summary

sweep dense 256 ok
sweep local 256 ok
sweep dilated 256 ok
sweep linear 256 ok
sweep hybrid 256 ok
sweep dense 1024 ok
sweep local 1024 ok
sweep dilated 1024 ok
sweep linear 1024 ok
sweep hybrid 1024 ok
sweep dense 4096 ok
sweep local 4096 ok
sweep dilated 4096 ok
sweep linear 4096 ok
sweep hybrid 4096 ok
sweep dense 8192 ok
sweep local 8192 ok
sweep dilated 8192 ok
sweep linear 8192 ok
sweep hybrid 8192 ok
wrote /kaggle/working/ATML/pa4/results/task2/part_b/summary.json


In [6]:
!zip -r /kaggle/working/task2_hybrid_results.zip results/task2/part_b -x 'results/task2/part_b/train_stream_*.npz' -x 'results/task2/part_b/*/model_update*.pt'

  adding: results/task2/part_b/ (stored 0%)
  adding: results/task2/part_b/hybrid_K16_mixed_seed6305/ (stored 0%)
  adding: results/task2/part_b/hybrid_K16_mixed_seed6305/result.json (deflated 85%)
  adding: results/task2/part_b/hybrid_K16_mixed_seed6305/test_queries.npz (deflated 8%)
  adding: results/task2/part_b/local_K16_mixed_seed6306/ (stored 0%)
  adding: results/task2/part_b/local_K16_mixed_seed6306/result.json (deflated 85%)
  adding: results/task2/part_b/local_K16_mixed_seed6306/test_queries.npz (deflated 8%)
  adding: results/task2/part_b/linear_K8_mixed_seed6305/ (stored 0%)
  adding: results/task2/part_b/linear_K8_mixed_seed6305/result.json (deflated 85%)
  adding: results/task2/part_b/linear_K8_mixed_seed6305/test_queries.npz (deflated 8%)
  adding: results/task2/part_b/local_K16_near_seed6306/ (stored 0%)
  adding: results/task2/part_b/local_K16_near_seed6306/result.json (deflated 85%)
  adding: results/task2/part_b/local_K16_near_seed6306/test_queries.npz (deflated 9%)
